# 01 - Extraction des caracteristiques

Ce notebook transforme chaque image du corpus en un vecteur de caracteristiques (SPAM, SRMQ1 ou SRM).

Il est prevu pour tourner sur un environnement a plusieurs coeurs, VPS ou Colab. L'extraction est
parallelisee et mise en cache, et reprend la ou elle s'est arretee en cas de coupure.

Toute la configuration se fait dans la premiere cellule. On change les valeurs, on ne touche pas au reste.

## Configuration

La seule cellule a modifier. Exemples : SPAM rapide sur tout, ou SRMQ1 sur les algorithmes adaptatifs.

In [ ]:
# ================= CONFIGURATION =================
FEATURE  = 'srmq1'                 # 'spam', 'srmq1' ou 'srm'
SOURCES  = ['natural', 'sd', 'sdxl', 'adm']
ALGOS    = ['uniward', 'hill']     # algorithmes a extraire ; [] pour covers seuls
PAYLOADS = [0.4]                   # charges utiles a extraire
INCLURE_COVER = True               # extraire aussi les images vierges
N_WORKERS = 0                      # 0 = tous les coeurs disponibles
SEED = 42
# ================================================
print('Configuration :', FEATURE, '| sources', SOURCES, '| algos', ALGOS, '| payloads', PAYLOADS)

## Environnement et corpus

Detecte Colab ou VPS. Sur VPS, place l'archive du corpus dans le dossier `memoire_data` a cote du
notebook, elle sera decompressee automatiquement.

In [ ]:
import os, glob, shutil, time
import numpy as np
np.random.seed(SEED)

# Chemins selon l'environnement
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
    ROOT = '/content/corpus'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')   # VPS ou local
    ROOT = os.path.abspath('./corpus')
os.makedirs(DATA_DIR, exist_ok=True)

FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
os.makedirs(FEAT_DIR, exist_ok=True)

# Restaure le corpus depuis la derniere archive si besoin
if not os.path.isdir(f'{ROOT}/natural/cover'):
    zips = sorted(glob.glob(f'{DATA_DIR}/corpus_*.zip'))
    if zips:
        print('Decompression du corpus depuis', zips[-1])
        shutil.unpack_archive(zips[-1], ROOT)
    else:
        print('Aucun corpus trouve. Placez une archive corpus_*.zip dans', DATA_DIR)

N_WORKERS = N_WORKERS or (os.cpu_count() or 1)
print('Coeurs utilises :', N_WORKERS, '| cache :', FEAT_DIR)

## Installation

In [ ]:
!pip install -q sealwatch imageio scipy
print('Installation terminee.')

## Fonction d'extraction

L'extracteur prend un tableau, pas un chemin, donc on charge l'image avant. Le test affiche la dimension
et le temps par image, ce qui permet d'estimer la duree totale.

In [ ]:
import imageio.v2 as imageio
import sealwatch as sw

def load_gray(path):
    x = np.asarray(imageio.imread(path))
    if x.ndim == 3:
        x = x[..., 0]
    return x

def to_vec(f):
    if isinstance(f, dict):
        return np.asarray(sw.tools.flatten(f), dtype=np.float32).ravel()
    return np.asarray(f, dtype=np.float32).ravel()

def _extracteur():
    if FEATURE == 'spam':
        return sw.spam
    if FEATURE == 'srmq1':
        return sw.srmq1
    return sw.srm

def extract(path):
    return to_vec(_extracteur().extract(load_gray(path)))

un = sorted(glob.glob(f'{ROOT}/natural/cover/*.pgm'))[:1]
if un:
    t = time.time(); d = extract(un[0]).shape[0]
    print(f'dimension {d}, temps par image {time.time()-t:.2f}s (avant parallelisation)')

## Extraction parallele avec cache reprenable

On traite les images par blocs, en parallele sur tous les coeurs, en sauvegardant apres chaque bloc.
Une coupure ne fait perdre au plus qu'un bloc. Relancez la cellule autant que necessaire.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def extract_set(paths, cache_file, bloc=200):
    if os.path.exists(cache_file):
        return np.load(cache_file)
    tmp = cache_file + '.part.npy'
    feats = list(np.load(tmp)) if os.path.exists(tmp) else []
    if feats:
        print('   reprise a', len(feats))
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        i = len(feats)
        while i < len(paths):
            lot = paths[i:i + bloc]
            feats.extend(ex.map(extract, lot, chunksize=4))
            i += len(lot)
            np.save(tmp, np.array(feats, dtype=np.float32))
            print('   ', i, '/', len(paths))
    arr = np.array(feats, dtype=np.float32)
    np.save(cache_file, arr)
    if os.path.exists(tmp):
        os.remove(tmp)
    return arr

def paths_of(source, setname):
    if setname == 'cover':
        return sorted(glob.glob(f'{ROOT}/{source}/cover/*.pgm'))
    algo, p = setname.split('_p')
    return sorted(glob.glob(f'{ROOT}/{source}/{algo}/*_p{p}.pgm'))

# Liste des ensembles a extraire, construite depuis la configuration
setnames = (['cover'] if INCLURE_COVER else []) + [f'{a}_p{p}' for a in ALGOS for p in PAYLOADS]
print('Ensembles vises :', setnames, '\n')

for source in SOURCES:
    for setname in setnames:
        paths = paths_of(source, setname)
        if not paths:
            print(f'{source} {setname}: aucune image, ignore'); continue
        cache_file = f'{FEAT_DIR}/{source}__{setname}.npy'
        if os.path.exists(cache_file):
            print(f'{source} {setname}: deja en cache'); continue
        t = time.time()
        print(f'{source} {setname}: {len(paths)} images')
        extract_set(paths, cache_file)
        print(f'   termine en {(time.time()-t)/60:.1f} min')
print('\nExtraction terminee ou reprise.')

## Verification

In [ ]:
caches = sorted(glob.glob(f'{FEAT_DIR}/*.npy'))
print(f'{len(caches)} fichiers dans {FEAT_DIR}\n')
for c in caches:
    a = np.load(c, mmap_mode='r')
    print(f'  {os.path.basename(c):24s} {a.shape}')
print(f'\nEnsembles vises cette execution : {len(SOURCES) * len(setnames)}')

## Suite

Recuperez le dossier des caracteristiques (sur Drive en Colab, ou par scp depuis le VPS) et pointez le
notebook 02 vers le meme `FEATURE`. Notez le temps par image et la duree totale dans le journal.

Pour lancer sur un VPS sans interface, voir le guide `VPS_SETUP.md` a la racine du depot.